# 📑 Hoist - EOM Litigation Report
 
## 🔗 Links
- **[BRD - EOM Litigation Report - TBD](tbd)**  
  Comprehensive documentation outlining the business requirements for this report.
 
- **[Story Ticket - TBD](tbd)**  
  Details about the story ticket, including tasks and progress.
 
- **[UAT Findings Ticket - TBD](TBD)**  
  A list of findings from UAT.
 
---

# Initialization

In [0]:
# %pip install openpyxl
# %restart_python

In [0]:
import os
import sys

dbutils.widgets.text("run_date", "")
dbutils.widgets.text("adhoc_run", '')

try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    repo_root = "/Workspace" + notebook_path.rsplit("/HoistInvestorReports/", 1)[0] 
    sys.path.insert(0, os.path.join(repo_root, "aws_utils"))
    sys.path.insert(0, os.path.join(repo_root, "01-utils"))
except Exception:
    sys.path.append("../../aws_utils")
    sys.path.append("../../01-utils")

from aws_utils import get_env_prefix, get_env_fullname
from helpers import load_configs, build_portfolio_filter

env = get_env_prefix()

In [0]:
from datetime import datetime
from pytz import timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window

athens_tz = timezone('Europe/Athens')
adhoc_run = dbutils.widgets.get("adhoc_run")

if adhoc_run == '1':
    dbutils.widgets.text("run_date", '')
    run_date = dbutils.widgets.get("run_date")
else:
    now_date = datetime.now(athens_tz).strftime('%Y-%m-%d')
    now_date_str = datetime.now(athens_tz).strftime('%Y%m%d')
    run_date = spark.sql(f"SELECT CurrentMonthLastWorkingDate FROM `{env}-cdh`.reference.ref_calendar WHERE CalendarDate = last_day(add_months(to_date('{now_date}'), -1))").first()["CurrentMonthLastWorkingDate"]

now_datetime_str = datetime.now(athens_tz).strftime('%Y%m%dT%H%M%S')

In [0]:
from typing import Any
from delta import DeltaTable  # type: ignore
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
    months_between,
    dateadd,
    datediff,
    regexp_replace
)

from typing import Optional
import delta  # type: ignore
from pyspark.sql import SparkSession
import pandas as pd

In [0]:
## Portfolios In Scope - All HOIST portfolios
PortfolioName = "('Cell', 'Orbit', 'Pearl Portfolio A', 'Mercury', 'Light')"    

# Load Views

In [0]:
dlegalprocessstatusagreement = spark.sql(f"""
       select *
       from `{env}-cdh`.derivative.dlegalprocessstatusagreement     
       where BusinessDate = '{run_date}' 
       AND PortfolioName in {PortfolioName}
       """)
dlegalprocessstatusagreement.createOrReplaceTempView("dlegalprocessstatusagreement")

ExchangeRates = spark.sql(f"""
       select *
       from `{env}-cdh`.reference.exchangerates 
       where BusinessDate = '{run_date}'
       """)
ExchangeRates.createOrReplaceTempView("ExchangeRates")

reassignments = spark.sql(f"""                             
        select distinct re.LoanID
        from `{env}-reportinputfiles`.investor.hoist_reassignments as re
          """)
reassignments.createOrReplaceTempView("reassignments")

dexposure = spark.sql(f"""
         SELECT dexp.*
         FROM `{env}-cdh`.derivative.dexposure dexp 
         LEFT JOIN reassignments as re on re.LoanID = dexp.ExposureKey
         WHERE sourcesystem = 'profits'
         AND dexp.PortfolioName in {PortfolioName}
         AND BusinessDate = '{run_date}'
         AND RE.LoanID IS NULL AND (dexp.JUSTIFICATIONCODE IS NULL OR dexp.JUSTIFICATIONCODE NOT IN (43055,43056))
         """)
dexposure.createOrReplaceTempView("dexposure")    

legalaction = spark.sql(f"""
         SELECT *
         FROM `{env}-cdh`.primary.legalaction
         WHERE sourcesystem = 'arotron'
         AND PortfolioName in {PortfolioName}
         AND BusinessDate = '{run_date}'
         """)
legalaction.createOrReplaceTempView("legalaction")    


# Dexposure

In [0]:
dexposure = spark.sql(f"""
SELECT 
AgreementKey, 
CDI, 
SUM (OutstandingTotalBalance) as OutstandingTotalBalance,
SUM(IFNULL(Principal,0) + IFNULL(DuePrincipal,0)) AS  PrincipalBalance,
SUM(IFNULL(Interest,0) + IFNULL(DueInterest,0) + IFNULL(PenaltyInterest,0) + IFNULL(UnrealizedInterest,0) + IFNULL(DueUnrealizedInterest,0) + IFNULL(PenaltyUnrealizedInterest,0)) AS Interest,
SUM(IFNULL(Expense,0) + IFNULL(DueExpense,0)) AS ExpensesFees
FROM dexposure
WHERE Isactiveflag = true
group by AgreementKey, CDI
""")

dexposure.createOrReplaceTempView("dexposure")

# Legal Action

In [0]:
Final = spark.sql(
f'''
SELECT DISTINCT 
act.PortfolioName,
act.AgreementKey,    
dexp.CDI,         
act.LegalActionType AS LegalAction, 
act.LegalType as RefersTo, 
cast(act.LegalActionDate as date) LegalActionDate,
act.LegalActionStepType as LegalPhase,
CAST(act.LegalActionStepStartDate AS DATE) as LegalPhaseStartDate,
act.LawyerPartyName LawOffice,
CAST(dexp.OutstandingTotalBalance AS DECIMAL(18,2)) AS OutstandingTotalBalance, 
CAST(dexp.PrincipalBalance AS DECIMAL(18,2)) AS PrincipalBalance,
CAST(dexp.Interest AS DECIMAL(18,2)) AS Interest,
CAST(dexp.ExpensesFees AS DECIMAL(18,2)) AS ExpensesFees,
lpsa.OperationalLegalStatusLevel3 LPS,
CAST(lpsa.LegalProcessStatusLastChangeDate AS DATE) AS LPSDate,
cast('{run_date}' as date) AS BusinessDate
FROM legalaction act
INNER JOIN dexposure AS dexp on act.agreementkey = dexp.agreementkey 
LEFT JOIN dlegalprocessstatusagreement as lpsa on lpsa.AgreementKey = dexp.AgreementKey
WHERE 1=1
AND LAST_DAY(CAST(act.LegalActionStepStartDate as Date))=LAST_DAY('{run_date}') 
AND (  (legalactiontype ='Αγωγή'and legaltype = 'Αγνώστου Διαμονής') OR
       (legalactiontype ='Αγωγή'and legaltype = 'Ελλειψεις Συμβατικων') OR
       (legalactiontype ='Αναγγελία'and legaltype = 'Πλειστηριασμό Ημών') OR
       (legalactiontype ='Αναγγελία'and legaltype = 'Πλειστηριασμό Τρίτου') OR
       (legalactiontype ='Β Επίδοση'and legaltype = 'Β'' Επίδοση Προς Γνώση') OR
       (legalactiontype ='Β Επίδοση'and legaltype = 'Διαταγή Πληρωμής') OR
       (legalactiontype ='Διαταγή Πληρωμής'and legaltype = 'Καταγγελία Σύμβασης') OR
       (legalactiontype ='Εγγραφή Βάρους'and legaltype = 'Αναγκαστική') OR
         --(legalactiontype ='Εγγραφή Βάρους'and legaltype = 'Συναινετική') OR -- excluded after talking with pasisis
       (legalactiontype ='Εξώδικο'and legaltype = 'Καταγγελία Σύμβασης') OR
       (legalactiontype ='Επίδοση ΔΠ'and legaltype = 'A'' Επίδοση Προς Γνώση') OR
       (legalactiontype ='Επίδοση ΔΠ'and legaltype = 'Α Επίδοση με Επιταγή') OR
       (legalactiontype ='Κατάσχεση κινητών-ακινήτων'and legaltype = 'Ακίνητα') OR
       (legalactiontype ='Κατάσχεση μισθωμάτων-εις χείρας Τρίτου'and legaltype = 'Ες Χείρας Τρίτου') OR
       (legalactiontype ='Πλειστηριασμός Ημών'and legaltype = 'Ακίνητης Ιδιοκτησίας - Ενυπόθηκο') OR
       (legalactiontype ='Πλειστηριασμός Τρίτου'and legaltype = 'Ακίνητης Ιδιοκτησίας - Ενυπόθηκο') OR
       (legalactiontype ='Πλειστηριασμός Τρίτου'and legaltype = 'Ακίνητης Ιδιοκτησίας - Μη ενυπόθηκο') )

'''
)
Final.createOrReplaceTempView("Final")

In [0]:
target_table_path_litigations = f"`{env}-cdh-delivery`.investorreports.hoist_Litigations"

spark.sql(f"""DELETE FROM {target_table_path_litigations} WHERE BusinessDate = '{run_date}' AND PortfolioName IN {PortfolioName}""")

Final = Final.withColumn("IngestionTimestamp", lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")))

Final.write \
        .format("delta") \
        .partitionBy("BusinessDate") \
        .mode("append") \
        .saveAsTable(target_table_path_litigations) # in order to write in delta_lake


spark.sql(f"REPAIR TABLE {target_table_path_litigations} SYNC METADATA")